# 08 TCI External Driver Factor Analysis

本 notebook 识别外部因素对贵金属动态风险连接度的解释力。与简单预测 TCI 水平不同，本 notebook 同时分析 TCI 水平、20 日变化、AR 残差和高连接状态，从而判断外部变量是否在控制 TCI 自身惯性后仍具有增量解释力。

解释原则：模型准确性以时间序列样本外表现为准；变量重要性只解释统计贡献，不直接解释为结构因果效应。


## 1. 环境、路径与参数


In [1]:
from pathlib import Path
import json
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import (
    ExtraTreesClassifier,
    ExtraTreesRegressor,
    RandomForestClassifier,
    RandomForestRegressor,
)
from sklearn.linear_model import ElasticNet, LogisticRegression, Ridge
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
DRIVER_DIR = DATA_DIR / "precious_metals_driver_csv_package"
PROCESSED_DIR = DATA_DIR / "processed"
COMPONENT_DIR = PROCESSED_DIR / "02_iceemdan_hierarchical_ward"
OUTPUT_DIR = PROCESSED_DIR / "08_tci_external_driver_factor_analysis"
FIG_DIR = OUTPUT_DIR / "visualizations"
for path in [OUTPUT_DIR, FIG_DIR]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 140
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
sns.set_style("whitegrid")

PARAMS = {
    "coverage_threshold": 0.80,
    "feature_lags": [1, 5, 20],
    "feature_rolling_windows": [20],
    "target_lags": [1, 5, 20],
    "delta_window": 20,
    "high_quantiles": {"high_q75": 0.75, "high_q90": 0.90},
    "n_cv_splits": 3,
    "min_initial_train_fraction": 0.55,
    "tree_estimators": 40,
    "tree_max_depth": 4,
    "tree_min_samples_leaf": 25,
    "permutation_repeats": 3,
    "random_state": 42,
}

MAIN_TARGET_COLS = [
    "tci_same_scale_STC",
    "tci_same_scale_MTC",
    "tci_within_gold",
    "tci_within_silver",
    "tci_within_platinum",
    "tci_within_palladium",
]
LTC_TARGET_COL = "tci_same_scale_LTC"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DRIVER_DIR exists:", DRIVER_DIR.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_ROOT: E:\@PythonProject\@Precious\precious-metal
DRIVER_DIR exists: True
OUTPUT_DIR: E:\@PythonProject\@Precious\precious-metal\data\processed\08_tci_external_driver_factor_analysis


C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\ensemble\weight_boosting.py:29: DeprecationWarning: numpy.core.umath_tests is an internal NumPy module and should not be imported. It will be removed in a future NumPy release.
  from numpy.core.umath_tests import inner1d


## 2. 数据读取与外部变量构造函数


In [2]:
def clean_numeric(series):
    return pd.to_numeric(series.astype(str).str.replace(",", ""), errors="coerce")


def read_supplemental_csv(file_name, value_col, new_col):
    path = DRIVER_DIR / file_name
    df = pd.read_csv(path)
    date_col = "observation_date" if "observation_date" in df.columns else "date"
    df = df[[date_col, value_col]].copy()
    df.columns = ["date", new_col]
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[new_col] = pd.to_numeric(df[new_col], errors="coerce")
    return df.dropna(subset=["date"]).sort_values("date")


def build_cftc_daily_features(daily_dates):
    cftc_path = DRIVER_DIR / "Disaggregated_-_Futures_Only_20260503.csv"
    cftc_usecols = [
        "Market_and_Exchange_Names",
        "Report_Date_as_YYYY_MM_DD",
        "Open_Interest_All",
        "Change_in_Open_Interest_All",
        "Prod_Merc_Positions_Long_All",
        "Prod_Merc_Positions_Short_All",
        "Swap_Positions_Long_All",
        "Swap__Positions_Short_All",
        "M_Money_Positions_Long_All",
        "M_Money_Positions_Short_All",
        "Other_Rept_Positions_Long_All",
        "Other_Rept_Positions_Short_All",
        "NonRept_Positions_Long_All",
        "NonRept_Positions_Short_All",
    ]
    market_to_metal = {
        "GOLD - COMMODITY EXCHANGE INC.": "gold",
        "SILVER - COMMODITY EXCHANGE INC.": "silver",
        "PLATINUM - NEW YORK MERCANTILE EXCHANGE": "platinum",
        "PALLADIUM - NEW YORK MERCANTILE EXCHANGE": "palladium",
    }
    cftc_raw = pd.read_csv(cftc_path, usecols=cftc_usecols, dtype=str)
    cftc_raw = cftc_raw[cftc_raw["Market_and_Exchange_Names"].isin(market_to_metal.keys())].copy()
    cftc_raw["metal"] = cftc_raw["Market_and_Exchange_Names"].map(market_to_metal)
    cftc_raw["report_date"] = pd.to_datetime(cftc_raw["Report_Date_as_YYYY_MM_DD"], errors="coerce")
    cftc_raw["available_date"] = cftc_raw["report_date"] + pd.Timedelta(days=3)

    numeric_cols = [col for col in cftc_usecols if col not in ["Market_and_Exchange_Names", "Report_Date_as_YYYY_MM_DD"]]
    for col in numeric_cols:
        cftc_raw[col] = clean_numeric(cftc_raw[col])
    cftc_raw = cftc_raw.dropna(subset=["available_date", "metal", "Open_Interest_All"])

    cftc_features = []
    for metal, group in cftc_raw.groupby("metal"):
        g = group.sort_values("available_date").copy()
        oi = g["Open_Interest_All"].replace(0, np.nan)
        out = pd.DataFrame({"date": g["available_date"].values})
        out["oi"] = oi.values
        out["oi_change_ratio"] = (g["Change_in_Open_Interest_All"] / oi).values
        out["managed_money_net_ratio"] = ((g["M_Money_Positions_Long_All"] - g["M_Money_Positions_Short_All"]) / oi).values
        out["producer_merchant_net_ratio"] = ((g["Prod_Merc_Positions_Long_All"] - g["Prod_Merc_Positions_Short_All"]) / oi).values
        out["swap_net_ratio"] = ((g["Swap_Positions_Long_All"] - g["Swap__Positions_Short_All"]) / oi).values
        out["other_reportable_net_ratio"] = ((g["Other_Rept_Positions_Long_All"] - g["Other_Rept_Positions_Short_All"]) / oi).values
        out["nonreportable_net_ratio"] = ((g["NonRept_Positions_Long_All"] - g["NonRept_Positions_Short_All"]) / oi).values
        cftc_features.append(out.set_index("date").add_prefix("cftc_{}_".format(metal)).reset_index())

    cftc_wide = cftc_features[0]
    for df in cftc_features[1:]:
        cftc_wide = cftc_wide.merge(df, on="date", how="outer")
    cftc_wide = cftc_wide.sort_values("date")
    cftc_daily = daily_dates.merge(cftc_wide, on="date", how="left").sort_values("date")
    cftc_cols = [col for col in cftc_daily.columns if col != "date"]
    cftc_daily[cftc_cols] = cftc_daily[cftc_cols].ffill()

    cftc_coverage = pd.DataFrame({
        "variable": cftc_cols,
        "non_missing": [int(cftc_daily[col].notnull().sum()) for col in cftc_cols],
        "coverage_ratio": [float(cftc_daily[col].notnull().mean()) for col in cftc_cols],
    })
    cftc_daily.to_csv(OUTPUT_DIR / "cftc_position_features_daily.csv", index=False, encoding="utf-8-sig")
    cftc_coverage.to_csv(OUTPUT_DIR / "cftc_feature_coverage.csv", index=False, encoding="utf-8-sig")
    return cftc_daily, cftc_coverage, len(cftc_raw)


def infer_feature_group(feature_name):
    name = feature_name.lower()
    if name.startswith("target_"):
        return "TCI_AR"
    if "dollar_index" in name:
        return "USD"
    if "yield_10y" in name or "real_rate" in name:
        return "RATE"
    if "breakeven" in name or "industrial_production" in name:
        return "INFLATION_GROWTH"
    if "vix" in name or "financial_stress" in name or "credit_spread" in name:
        return "RISK_STRESS"
    if "wti" in name:
        return "OIL"
    if "gpr" in name:
        return "GPR"
    if "gold_silver" in name:
        return "REL_GOLD_SILVER"
    if "platinum_palladium" in name:
        return "REL_PLATINUM_PALLADIUM"
    if "gold_platinum" in name:
        return "REL_GOLD_PLATINUM"
    if "cftc" in name:
        if "gold" in name:
            return "CFTC_GOLD"
        if "silver" in name:
            return "CFTC_SILVER"
        if "platinum" in name:
            return "CFTC_PLATINUM"
        if "palladium" in name:
            return "CFTC_PALLADIUM"
        return "CFTC"
    return "OTHER"


## 3. LTC 窗口敏感性函数


In [3]:
def total_r2_tci_from_values(values):
    values = np.asarray(values, dtype=float)
    corr = np.corrcoef(values, rowvar=False)
    corr = np.nan_to_num(corr)
    k = corr.shape[0]
    total_r2 = []
    for target_idx in range(k):
        pred_idx = [idx for idx in range(k) if idx != target_idx]
        rxx = corr[np.ix_(pred_idx, pred_idx)]
        ryx = corr[pred_idx, target_idx]
        try:
            r2_value = float(ryx.dot(np.linalg.solve(rxx, ryx)))
        except Exception:
            r2_value = float(ryx.dot(np.linalg.pinv(rxx)).dot(ryx))
        total_r2.append(float(np.clip(r2_value, 0.0, 1.0)))
    return float(np.mean(total_r2) * 100.0), float(np.min(total_r2) * 100.0), float(np.max(total_r2) * 100.0)


def compute_ltc_window_sensitivity():
    components = pd.read_csv(COMPONENT_DIR / "hierarchical_ward_components_wide.csv", parse_dates=["date"])
    ltc_cols = [col for col in components.columns if col.endswith("_LTC")]
    ltc_values = components[ltc_cols].values
    rows = []
    for window in [252, 300, 500, 756, 1000, 1500, 2000, 3100]:
        if window > len(components):
            continue
        tci_values, min_values, max_values = [], [], []
        for end_idx in range(window - 1, len(components)):
            start_idx = end_idx - window + 1
            mean_tci, min_tci, max_tci = total_r2_tci_from_values(ltc_values[start_idx:end_idx + 1])
            tci_values.append(mean_tci)
            min_values.append(min_tci)
            max_values.append(max_tci)
        arr = np.asarray(tci_values)
        rows.append({
            "window_size": window,
            "n_windows": len(arr),
            "mean_tci": float(np.mean(arr)),
            "std_tci": float(np.std(arr)),
            "min_tci": float(np.min(arr)),
            "q25_tci": float(np.percentile(arr, 25)),
            "median_tci": float(np.percentile(arr, 50)),
            "q75_tci": float(np.percentile(arr, 75)),
            "max_tci": float(np.max(arr)),
            "mean_min_target_r2": float(np.mean(min_values)),
            "mean_max_target_r2": float(np.mean(max_values)),
        })
    sensitivity = pd.DataFrame(rows)
    sensitivity.to_csv(OUTPUT_DIR / "ltc_window_sensitivity.csv", index=False, encoding="utf-8-sig")

    fig, ax = plt.subplots(figsize=(8.2, 4.8))
    ax.plot(sensitivity["window_size"], sensitivity["mean_tci"], marker="o", linewidth=2.0, label="Mean TCI")
    ax.fill_between(sensitivity["window_size"].values, sensitivity["q25_tci"].values, sensitivity["q75_tci"].values, alpha=0.20, label="IQR")
    ax.set_xlabel("Rolling window size")
    ax.set_ylabel("LTC same-scale TCI (%)")
    ax.set_title("LTC TCI sensitivity to rolling-window length")
    ax.set_ylim(60, 101)
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "ltc_window_sensitivity.png")
    fig.savefig(FIG_DIR / "ltc_window_sensitivity.pdf")
    plt.close(fig)
    return sensitivity


## 4. 特征工程与模型工具函数


In [4]:
def build_external_features(df, raw_cols, lags, rolling_windows):
    features = pd.DataFrame({"date": df["date"]})
    engineered_names = []
    for col in raw_cols:
        series = pd.to_numeric(df[col], errors="coerce")
        for lag in lags:
            new_col = "{}_lag{}".format(col, lag)
            features[new_col] = series.shift(lag)
            engineered_names.append(new_col)
        for window in rolling_windows:
            min_periods = max(5, int(window / 4))
            mean_col = "{}_ma{}_lag1".format(col, window)
            std_col = "{}_std{}_lag1".format(col, window)
            features[mean_col] = series.rolling(window, min_periods=min_periods).mean().shift(1)
            features[std_col] = series.rolling(window, min_periods=min_periods).std().shift(1)
            engineered_names.extend([mean_col, std_col])
    return features, engineered_names


def add_variant_lags(df, y_col, prefix):
    out = pd.DataFrame({"date": df["date"]})
    y = pd.to_numeric(df[y_col], errors="coerce")
    lag_cols = []
    for lag in PARAMS["target_lags"]:
        col = "{}_lag{}".format(prefix, lag)
        out[col] = y.shift(lag)
        lag_cols.append(col)
    ma_col = "{}_ma20_lag1".format(prefix)
    diff_col = "{}_diff1_lag1".format(prefix)
    out[ma_col] = y.rolling(20, min_periods=5).mean().shift(1)
    out[diff_col] = y.diff().shift(1)
    lag_cols.extend([ma_col, diff_col])
    return out, lag_cols


def time_series_splits(n_obs, n_splits=3, min_initial_fraction=0.55):
    min_train = int(np.floor(n_obs * min_initial_fraction))
    remaining = n_obs - min_train
    test_size = int(np.floor(remaining / n_splits))
    splits = []
    for fold in range(n_splits):
        train_end = min_train + fold * test_size
        test_start = train_end
        test_end = n_obs if fold == n_splits - 1 else test_start + test_size
        if test_end > test_start and train_end > 50:
            splits.append((fold + 1, np.arange(0, train_end), np.arange(test_start, test_end)))
    return splits


def make_reg_model(model_name):
    if model_name == "Ridge":
        return Pipeline([("scale", StandardScaler()), ("model", Ridge(alpha=10.0))])
    if model_name == "ElasticNet":
        return Pipeline([("scale", StandardScaler()), ("model", ElasticNet(alpha=0.01, l1_ratio=0.3, max_iter=20000))])
    if model_name == "RandomForest":
        return RandomForestRegressor(
            n_estimators=PARAMS["tree_estimators"],
            max_depth=PARAMS["tree_max_depth"],
            min_samples_leaf=PARAMS["tree_min_samples_leaf"],
            max_features="sqrt",
            random_state=PARAMS["random_state"],
            n_jobs=-1,
        )
    if model_name == "ExtraTrees":
        return ExtraTreesRegressor(
            n_estimators=PARAMS["tree_estimators"],
            max_depth=PARAMS["tree_max_depth"],
            min_samples_leaf=PARAMS["tree_min_samples_leaf"],
            max_features="sqrt",
            random_state=PARAMS["random_state"],
            n_jobs=-1,
        )
    raise ValueError("Unknown regression model: {}".format(model_name))


def make_clf_model(model_name):
    if model_name == "Logistic":
        return Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=PARAMS["random_state"])),
        ])
    if model_name == "RandomForestClassifier":
        return RandomForestClassifier(
            n_estimators=PARAMS["tree_estimators"],
            max_depth=PARAMS["tree_max_depth"],
            min_samples_leaf=PARAMS["tree_min_samples_leaf"],
            max_features="sqrt",
            class_weight="balanced",
            random_state=PARAMS["random_state"],
            n_jobs=-1,
        )
    if model_name == "ExtraTreesClassifier":
        return ExtraTreesClassifier(
            n_estimators=PARAMS["tree_estimators"],
            max_depth=PARAMS["tree_max_depth"],
            min_samples_leaf=PARAMS["tree_min_samples_leaf"],
            max_features="sqrt",
            class_weight="balanced",
            random_state=PARAMS["random_state"],
            n_jobs=-1,
        )
    raise ValueError("Unknown classifier model: {}".format(model_name))


def regression_metrics(y_true, y_pred):
    return {
        "r2": float(r2_score(y_true, y_pred)),
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "test_std": float(np.std(y_true)),
    }


def balanced_accuracy_manual(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    recalls = []
    for cls in [0, 1]:
        mask = y_true == cls
        if mask.sum() > 0:
            recalls.append(float((y_pred[mask] == cls).mean()))
    return float(np.mean(recalls)) if recalls else np.nan


def classification_metrics(y_true, y_score, threshold=0.5):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score, dtype=float)
    y_pred = (y_score >= threshold).astype(int)
    if len(np.unique(y_true)) < 2:
        auc = np.nan
        ap = np.nan
    else:
        auc = float(roc_auc_score(y_true, y_score))
        ap = float(average_precision_score(y_true, y_score))
    return {
        "auc": auc,
        "average_precision": ap,
        "balanced_accuracy": balanced_accuracy_manual(y_true, y_pred),
        "f1": float(f1_score(y_true, y_pred)) if y_pred.sum() + y_true.sum() > 0 else 0.0,
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "positive_rate_test": float(y_true.mean()),
        "positive_rate_pred": float(y_pred.mean()),
    }


def safe_predict_proba(model, x_values, fallback_prob):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(x_values)[:, 1]
    return np.repeat(fallback_prob, x_values.shape[0])


## 5. 重要性评分函数


In [5]:
def grouped_permutation_importance_regression(model, x_test, y_test, feature_cols, group_map, n_repeats=3, random_state=42):
    rng = np.random.RandomState(random_state)
    x_values = np.asarray(x_test).copy()
    base_score = float(r2_score(y_test, model.predict(x_values)))
    group_to_indices = {}
    for idx, col in enumerate(feature_cols):
        group_to_indices.setdefault(group_map.get(col, infer_feature_group(col)), []).append(idx)
    rows = []
    for group, indices in sorted(group_to_indices.items()):
        scores = []
        for _ in range(n_repeats):
            x_perm = x_values.copy()
            perm = rng.permutation(x_perm.shape[0])
            x_perm[:, indices] = x_perm[perm][:, indices]
            scores.append(float(r2_score(y_test, model.predict(x_perm))))
        scores = np.asarray(scores)
        rows.append({
            "model_type": "RandomForestPermutation",
            "group": group,
            "base_score": base_score,
            "permuted_score_mean": float(scores.mean()),
            "permuted_score_std": float(scores.std()),
            "importance_drop": float(base_score - scores.mean()),
            "importance_positive": float(max(0.0, base_score - scores.mean())),
            "n_features": len(indices),
        })
    return pd.DataFrame(rows)


def grouped_permutation_importance_classification(model, x_test, y_test, feature_cols, group_map, n_repeats=3, random_state=42):
    rng = np.random.RandomState(random_state)
    x_values = np.asarray(x_test).copy()
    if len(np.unique(y_test)) < 2:
        base_score = np.nan
    else:
        base_score = float(roc_auc_score(y_test, model.predict_proba(x_values)[:, 1]))
    group_to_indices = {}
    for idx, col in enumerate(feature_cols):
        group_to_indices.setdefault(group_map.get(col, infer_feature_group(col)), []).append(idx)
    rows = []
    for group, indices in sorted(group_to_indices.items()):
        scores = []
        for _ in range(n_repeats):
            x_perm = x_values.copy()
            perm = rng.permutation(x_perm.shape[0])
            x_perm[:, indices] = x_perm[perm][:, indices]
            if len(np.unique(y_test)) < 2:
                scores.append(np.nan)
            else:
                scores.append(float(roc_auc_score(y_test, model.predict_proba(x_perm)[:, 1])))
        scores = np.asarray(scores, dtype=float)
        mean_score = float(np.nanmean(scores)) if np.isfinite(scores).any() else np.nan
        drop = base_score - mean_score if np.isfinite(base_score) and np.isfinite(mean_score) else np.nan
        rows.append({
            "model_type": "RandomForestClassifierPermutation",
            "group": group,
            "base_score": base_score,
            "permuted_score_mean": mean_score,
            "permuted_score_std": float(np.nanstd(scores)) if np.isfinite(scores).any() else np.nan,
            "importance_drop": float(drop) if np.isfinite(drop) else np.nan,
            "importance_positive": float(max(0.0, drop)) if np.isfinite(drop) else np.nan,
            "n_features": len(indices),
        })
    return pd.DataFrame(rows)


def elasticnet_group_scores(model, feature_cols, group_map):
    coefs = model.named_steps["model"].coef_
    rows = []
    for feature, coef in zip(feature_cols, coefs):
        rows.append({
            "model_type": "ElasticNetCoefficient",
            "feature": feature,
            "group": group_map.get(feature, infer_feature_group(feature)),
            "coefficient": float(coef),
            "abs_coefficient": float(abs(coef)),
        })
    raw = pd.DataFrame(rows)
    group = raw.groupby("group").agg({"abs_coefficient": "sum", "coefficient": "sum", "feature": "count"}).reset_index()
    group = group.rename(columns={"abs_coefficient": "importance_positive", "coefficient": "signed_importance", "feature": "n_features"})
    group["model_type"] = "ElasticNetCoefficient"
    group["base_score"] = np.nan
    group["permuted_score_mean"] = np.nan
    group["permuted_score_std"] = np.nan
    group["importance_drop"] = group["importance_positive"]
    return group


def logistic_group_scores(model, feature_cols, group_map):
    coefs = model.named_steps["model"].coef_[0]
    rows = []
    for feature, coef in zip(feature_cols, coefs):
        rows.append({
            "model_type": "LogisticCoefficient",
            "feature": feature,
            "group": group_map.get(feature, infer_feature_group(feature)),
            "coefficient": float(coef),
            "abs_coefficient": float(abs(coef)),
        })
    raw = pd.DataFrame(rows)
    group = raw.groupby("group").agg({"abs_coefficient": "sum", "coefficient": "sum", "feature": "count"}).reset_index()
    group = group.rename(columns={"abs_coefficient": "importance_positive", "coefficient": "signed_importance", "feature": "n_features"})
    group["model_type"] = "LogisticCoefficient"
    group["base_score"] = np.nan
    group["permuted_score_mean"] = np.nan
    group["permuted_score_std"] = np.nan
    group["importance_drop"] = group["importance_positive"]
    return group


## 6. 回归与分类建模函数


In [6]:
def summarize_regression(regression_rows):
    detail = pd.DataFrame(regression_rows)
    summary = detail.groupby(["target", "target_type", "model_family", "model", "feature_set"])[["r2", "rmse", "mae", "test_std"]].agg(["mean", "std"])
    summary.columns = ["{}_{}".format(a, b) for a, b in summary.columns]
    summary = summary.reset_index()
    return detail, summary


def run_level_or_delta_regression(model_base, target_col, target_type, y_col, engineered_external_cols, feature_group_map):
    lag_df, lag_cols = add_variant_lags(model_base, y_col, "target_{}_{}".format(target_type, target_col))
    data = model_base[["date", y_col] + engineered_external_cols].merge(lag_df, on="date", how="left")
    data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    rows, importance_rows = [], []
    model_names = ["Ridge", "ElasticNet", "RandomForest", "ExtraTrees"]

    for fold, train_idx, test_idx in time_series_splits(len(data), PARAMS["n_cv_splits"], PARAMS["min_initial_train_fraction"]):
        train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()
        y_train, y_test = train[y_col].values, test[y_col].values

        mean_pred = np.repeat(np.mean(y_train), len(y_test))
        rows.append({"target": target_col, "target_type": target_type, "fold": fold, "model_family": "Mean", "model": "Mean", "feature_set": "mean_baseline", "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **regression_metrics(y_test, mean_pred)})

        ar_model = make_reg_model("Ridge")
        ar_model.fit(train[lag_cols].values, y_train)
        ar_pred = ar_model.predict(test[lag_cols].values)
        rows.append({"target": target_col, "target_type": target_type, "fold": fold, "model_family": "AR", "model": "AR_Ridge", "feature_set": "target_lags", "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **regression_metrics(y_test, ar_pred)})

        for model_name in model_names:
            for family, feature_cols in [("Exog", engineered_external_cols), ("ARX", engineered_external_cols + lag_cols)]:
                model = make_reg_model(model_name)
                model.fit(train[feature_cols].values, y_train)
                pred = model.predict(test[feature_cols].values)
                rows.append({"target": target_col, "target_type": target_type, "fold": fold, "model_family": family, "model": "{}_{}".format(family, model_name), "feature_set": "external_only" if family == "Exog" else "arx_external_plus_target_lags", "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **regression_metrics(y_test, pred)})

        if fold == time_series_splits(len(data), PARAMS["n_cv_splits"], PARAMS["min_initial_train_fraction"])[-1][0]:
            group_map = dict(feature_group_map)
            for col in lag_cols:
                group_map[col] = "TCI_AR"
            for mode, feature_cols in [("external_only", engineered_external_cols), ("arx_external_plus_target_lags", engineered_external_cols + lag_cols)]:
                enet = make_reg_model("ElasticNet")
                enet.fit(train[feature_cols].values, y_train)
                enet_groups = elasticnet_group_scores(enet, feature_cols, group_map)
                enet_groups.insert(0, "target", target_col)
                enet_groups.insert(1, "target_type", target_type)
                enet_groups.insert(2, "mode", mode)
                enet_groups.insert(3, "fold", fold)
                importance_rows.append(enet_groups)
                rf = make_reg_model("RandomForest")
                rf.fit(train[feature_cols].values, y_train)
                rf_groups = grouped_permutation_importance_regression(rf, test[feature_cols].values, y_test, feature_cols, group_map, PARAMS["permutation_repeats"], PARAMS["random_state"])
                rf_groups.insert(0, "target", target_col)
                rf_groups.insert(1, "target_type", target_type)
                rf_groups.insert(2, "mode", mode)
                rf_groups.insert(3, "fold", fold)
                importance_rows.append(rf_groups)
    return rows, importance_rows


def run_ar_residual_regression(model_base, target_col, engineered_external_cols, feature_group_map):
    y_col = target_col
    lag_df, lag_cols = add_variant_lags(model_base, y_col, "target_ar_residual_{}".format(target_col))
    data = model_base[["date", y_col] + engineered_external_cols].merge(lag_df, on="date", how="left")
    data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    rows, residual_rows, importance_rows = [], [], []
    model_names = ["Ridge", "ElasticNet", "RandomForest", "ExtraTrees"]
    splits = time_series_splits(len(data), PARAMS["n_cv_splits"], PARAMS["min_initial_train_fraction"])

    for fold, train_idx, test_idx in splits:
        train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()
        y_train, y_test = train[y_col].values, test[y_col].values
        ar_model = make_reg_model("Ridge")
        ar_model.fit(train[lag_cols].values, y_train)
        ar_pred_train = ar_model.predict(train[lag_cols].values)
        ar_pred_test = ar_model.predict(test[lag_cols].values)
        resid_train = y_train - ar_pred_train
        resid_test = y_test - ar_pred_test

        ar_metrics = regression_metrics(y_test, ar_pred_test)
        rows.append({"target": target_col, "target_type": "ar_residual", "fold": fold, "model_family": "AR", "model": "AR_Ridge", "feature_set": "target_lags", "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **ar_metrics})

        mean_resid = np.repeat(np.mean(resid_train), len(resid_test))
        synthetic_mean = ar_pred_test + mean_resid
        residual_rows.append({"target": target_col, "fold": fold, "model_family": "Mean", "model": "Residual_Mean", "feature_set": "residual_mean", **{"residual_" + k: v for k, v in regression_metrics(resid_test, mean_resid).items()}, **{"synthetic_" + k: v for k, v in regression_metrics(y_test, synthetic_mean).items()}, "ar_r2": ar_metrics["r2"], "synthetic_incremental_r2_vs_ar": regression_metrics(y_test, synthetic_mean)["r2"] - ar_metrics["r2"]})

        for model_name in model_names:
            for family, feature_cols in [("Exog", engineered_external_cols), ("ARX", engineered_external_cols + lag_cols)]:
                model = make_reg_model(model_name)
                model.fit(train[feature_cols].values, resid_train)
                resid_pred = model.predict(test[feature_cols].values)
                synthetic_pred = ar_pred_test + resid_pred
                res_metrics = regression_metrics(resid_test, resid_pred)
                syn_metrics = regression_metrics(y_test, synthetic_pred)
                residual_rows.append({"target": target_col, "fold": fold, "model_family": family, "model": "Residual_{}_{}".format(family, model_name), "feature_set": "external_only" if family == "Exog" else "arx_external_plus_target_lags", **{"residual_" + k: v for k, v in res_metrics.items()}, **{"synthetic_" + k: v for k, v in syn_metrics.items()}, "ar_r2": ar_metrics["r2"], "synthetic_incremental_r2_vs_ar": syn_metrics["r2"] - ar_metrics["r2"]})
                rows.append({"target": target_col, "target_type": "ar_residual", "fold": fold, "model_family": family, "model": "{}_{}".format(family, model_name), "feature_set": "external_only" if family == "Exog" else "arx_external_plus_target_lags", "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **res_metrics})

        if fold == splits[-1][0]:
            group_map = dict(feature_group_map)
            for col in lag_cols:
                group_map[col] = "TCI_AR"
            for mode, feature_cols in [("external_only", engineered_external_cols), ("arx_external_plus_target_lags", engineered_external_cols + lag_cols)]:
                enet = make_reg_model("ElasticNet")
                enet.fit(train[feature_cols].values, resid_train)
                enet_groups = elasticnet_group_scores(enet, feature_cols, group_map)
                enet_groups.insert(0, "target", target_col)
                enet_groups.insert(1, "target_type", "ar_residual")
                enet_groups.insert(2, "mode", mode)
                enet_groups.insert(3, "fold", fold)
                importance_rows.append(enet_groups)
                rf = make_reg_model("RandomForest")
                rf.fit(train[feature_cols].values, resid_train)
                rf_groups = grouped_permutation_importance_regression(rf, test[feature_cols].values, resid_test, feature_cols, group_map, PARAMS["permutation_repeats"], PARAMS["random_state"])
                rf_groups.insert(0, "target", target_col)
                rf_groups.insert(1, "target_type", "ar_residual")
                rf_groups.insert(2, "mode", mode)
                rf_groups.insert(3, "fold", fold)
                importance_rows.append(rf_groups)
    return rows, residual_rows, importance_rows


def run_high_state_classification(model_base, target_col, engineered_external_cols, feature_group_map):
    lag_df, lag_cols = add_variant_lags(model_base, target_col, "target_high_state_{}".format(target_col))
    data = model_base[["date", target_col] + engineered_external_cols].merge(lag_df, on="date", how="left")
    data = data.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    rows, threshold_rows, importance_rows = [], [], []
    classifier_specs = [
        ("AR", "AR_Logistic", "Logistic", lag_cols),
        ("Exog", "Exog_Logistic", "Logistic", engineered_external_cols),
        ("Exog", "Exog_RandomForestClassifier", "RandomForestClassifier", engineered_external_cols),
        ("Exog", "Exog_ExtraTreesClassifier", "ExtraTreesClassifier", engineered_external_cols),
        ("ARX", "ARX_Logistic", "Logistic", engineered_external_cols + lag_cols),
        ("ARX", "ARX_RandomForestClassifier", "RandomForestClassifier", engineered_external_cols + lag_cols),
        ("ARX", "ARX_ExtraTreesClassifier", "ExtraTreesClassifier", engineered_external_cols + lag_cols),
    ]
    splits = time_series_splits(len(data), PARAMS["n_cv_splits"], PARAMS["min_initial_train_fraction"])

    for high_label, quantile in PARAMS["high_quantiles"].items():
        for fold, train_idx, test_idx in splits:
            train, test = data.iloc[train_idx].copy(), data.iloc[test_idx].copy()
            threshold_value = float(train[target_col].quantile(quantile))
            y_train = (train[target_col].values >= threshold_value).astype(int)
            y_test = (test[target_col].values >= threshold_value).astype(int)
            prevalence = float(y_train.mean())
            threshold_rows.append({"target": target_col, "high_state": high_label, "fold": fold, "quantile": quantile, "threshold": threshold_value, "train_positive_rate": prevalence, "test_positive_rate": float(y_test.mean())})

            base_score = np.repeat(prevalence, len(y_test))
            rows.append({"target": target_col, "high_state": high_label, "fold": fold, "model_family": "Prevalence", "model": "Prevalence", "feature_set": "prevalence", "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **classification_metrics(y_test, base_score)})

            for family, label, model_name, feature_cols in classifier_specs:
                if len(np.unique(y_train)) < 2:
                    y_score = base_score
                else:
                    model = make_clf_model(model_name)
                    model.fit(train[feature_cols].values, y_train)
                    y_score = safe_predict_proba(model, test[feature_cols].values, prevalence)
                rows.append({"target": target_col, "high_state": high_label, "fold": fold, "model_family": family, "model": label, "feature_set": "target_lags" if family == "AR" else ("external_only" if family == "Exog" else "arx_external_plus_target_lags"), "train_start": train["date"].min(), "train_end": train["date"].max(), "test_start": test["date"].min(), "test_end": test["date"].max(), "n_train": len(train), "n_test": len(test), **classification_metrics(y_test, y_score)})

            if fold == splits[-1][0] and len(np.unique(y_train)) >= 2:
                group_map = dict(feature_group_map)
                for col in lag_cols:
                    group_map[col] = "TCI_AR"
                for mode, feature_cols in [("external_only", engineered_external_cols), ("arx_external_plus_target_lags", engineered_external_cols + lag_cols)]:
                    logit = make_clf_model("Logistic")
                    logit.fit(train[feature_cols].values, y_train)
                    logit_groups = logistic_group_scores(logit, feature_cols, group_map)
                    logit_groups.insert(0, "target", target_col)
                    logit_groups.insert(1, "high_state", high_label)
                    logit_groups.insert(2, "mode", mode)
                    logit_groups.insert(3, "fold", fold)
                    importance_rows.append(logit_groups)
                    rf = make_clf_model("RandomForestClassifier")
                    rf.fit(train[feature_cols].values, y_train)
                    rf_groups = grouped_permutation_importance_classification(rf, test[feature_cols].values, y_test, feature_cols, group_map, PARAMS["permutation_repeats"], PARAMS["random_state"])
                    rf_groups.insert(0, "target", target_col)
                    rf_groups.insert(1, "high_state", high_label)
                    rf_groups.insert(2, "mode", mode)
                    rf_groups.insert(3, "fold", fold)
                    importance_rows.append(rf_groups)
    return rows, threshold_rows, importance_rows


## 7. 可视化函数


In [7]:
GROUP_ORDER = [
    "USD",
    "RATE",
    "INFLATION_GROWTH",
    "RISK_STRESS",
    "OIL",
    "GPR",
    "REL_GOLD_SILVER",
    "REL_PLATINUM_PALLADIUM",
    "REL_GOLD_PLATINUM",
    "CFTC_GOLD",
    "CFTC_SILVER",
    "CFTC_PLATINUM",
    "CFTC_PALLADIUM",
]


def normalized_group_matrix(df, row_col, group_col="group", value_col="importance_positive"):
    data = df.copy()
    data = data[data[group_col] != "TCI_AR"].copy()
    data[value_col] = pd.to_numeric(data[value_col], errors="coerce").fillna(0.0)
    grouped = data.groupby([row_col, group_col])[value_col].sum().reset_index()
    grouped["row_total"] = grouped.groupby(row_col)[value_col].transform("sum")
    grouped["normalized"] = np.where(grouped["row_total"] > 0, grouped[value_col] / grouped["row_total"], 0.0)
    matrix = grouped.pivot(index=row_col, columns=group_col, values="normalized").fillna(0.0)
    matrix = matrix[[group for group in GROUP_ORDER if group in matrix.columns]]
    return matrix


def save_heatmap(matrix, title, filename, cmap="YlGnBu", center=None, fmt=".2f"):
    fig_w = max(8.0, 0.8 * max(1, matrix.shape[1]) + 3.0)
    fig_h = max(4.8, 0.55 * max(1, matrix.shape[0]) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_w, fig_h))
    sns.heatmap(matrix, annot=True, fmt=fmt, cmap=cmap, center=center, linewidths=0.4, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("")
    fig.tight_layout()
    fig.savefig(FIG_DIR / "{}.png".format(filename))
    fig.savefig(FIG_DIR / "{}.pdf".format(filename))
    plt.close(fig)


def plot_analysis_outputs(regression_summary, regression_incremental, residual_summary, classification_summary, driver_group_summary):
    selected = regression_incremental[
        regression_incremental["model"].isin(["Exog_ElasticNet", "ARX_ElasticNet", "Exog_RandomForest", "ARX_RandomForest"])
        & regression_incremental["target_type"].isin(["level", "delta20"])
    ].copy()
    selected["column"] = selected["target_type"] + ":" + selected["model"]
    matrix = selected.pivot(index="target", columns="column", values="incremental_r2_vs_ar").fillna(0.0)
    save_heatmap(matrix, "Incremental R2 relative to AR baseline", "regression_incremental_r2_heatmap", cmap="RdYlGn", center=0)

    delta_imp = driver_group_summary[
        (driver_group_summary["analysis_type"] == "regression")
        & (driver_group_summary["target_type"] == "delta20")
        & (driver_group_summary["mode"] == "arx_external_plus_target_lags")
        & (driver_group_summary["model_type"] == "ElasticNetCoefficient")
    ].copy()
    save_heatmap(normalized_group_matrix(delta_imp, "target"), "Delta20 external driver group scores", "delta20_driver_group_heatmap", cmap="PuBuGn")

    residual_imp = driver_group_summary[
        (driver_group_summary["analysis_type"] == "regression")
        & (driver_group_summary["target_type"] == "ar_residual")
        & (driver_group_summary["mode"] == "external_only")
        & (driver_group_summary["model_type"] == "ElasticNetCoefficient")
    ].copy()
    save_heatmap(normalized_group_matrix(residual_imp, "target"), "AR residual external driver group scores", "ar_residual_driver_group_heatmap", cmap="PuBuGn")

    auc_data = classification_summary[
        classification_summary["model"].isin(["Prevalence", "AR_Logistic", "Exog_Logistic", "ARX_Logistic", "Exog_RandomForestClassifier", "ARX_RandomForestClassifier"])
    ].copy()
    auc_data["column"] = auc_data["high_state"] + ":" + auc_data["model"]
    auc_matrix = auc_data.pivot(index="target", columns="column", values="auc_mean")
    save_heatmap(auc_matrix, "High connectedness state AUC", "high_state_auc_heatmap", cmap="RdYlGn", center=0.5)

    high_imp = driver_group_summary[
        (driver_group_summary["analysis_type"] == "classification")
        & (driver_group_summary["mode"] == "arx_external_plus_target_lags")
        & (driver_group_summary["model_type"] == "LogisticCoefficient")
    ].copy()
    high_imp["row"] = high_imp["target"] + ":" + high_imp["high_state"]
    save_heatmap(normalized_group_matrix(high_imp, "row"), "High-state external driver group scores", "high_state_driver_group_heatmap", cmap="YlGnBu")


## 8. 主分析流程


In [8]:
def run_analysis():
    feature_panel_path = DRIVER_DIR / "feature_panel_aligned_to_dynamic_tci_252.csv"
    base_panel = pd.read_csv(feature_panel_path, parse_dates=["date"]).sort_values("date").reset_index(drop=True)
    target_cols_all = [col for col in base_panel.columns if col.startswith("tci_")]
    daily_dates = base_panel[["date"]].copy()

    supplemental_specs = [
        ("St.-Louis-Fed-Financial-Stress-Index(STLFSI4).csv", "STLFSI4", "financial_stress"),
        ("Industrial-Production-Total-Index(INDPRO).csv", "INDPRO", "industrial_production"),
        ("ICE-BofA-US-Corporate-Index-Option-Adjusted-Spread-(BAMLC0A0CM).csv", "BAMLC0A0CM", "credit_spread_ig"),
        ("ICE-BofA-US-High-Yield-Index-Option-Adjusted-Spread(BAMLH0A0HYM2).csv", "BAMLH0A0HYM2", "credit_spread_hy"),
    ]
    supplemental_daily = daily_dates.copy()
    for file_name, value_col, new_col in supplemental_specs:
        raw = read_supplemental_csv(file_name, value_col, new_col)
        tmp = daily_dates.merge(raw, on="date", how="left")
        tmp[new_col] = tmp[new_col].ffill()
        supplemental_daily = supplemental_daily.merge(tmp[["date", new_col]], on="date", how="left")
    supplemental_coverage = pd.DataFrame([{
        "variable": col,
        "non_missing": int(supplemental_daily[col].notnull().sum()),
        "coverage_ratio": float(supplemental_daily[col].notnull().mean()),
    } for col in supplemental_daily.columns if col != "date"])
    supplemental_coverage.to_csv(OUTPUT_DIR / "supplemental_feature_coverage.csv", index=False, encoding="utf-8-sig")

    cftc_daily, cftc_coverage, cftc_precious_rows = build_cftc_daily_features(daily_dates)
    full_panel = base_panel.merge(supplemental_daily, on="date", how="left")
    full_panel = full_panel.merge(cftc_daily, on="date", how="left").sort_values("date").reset_index(drop=True)

    candidate_feature_cols = [col for col in full_panel.columns if col not in ["date"] + target_cols_all]
    feature_coverage = pd.DataFrame([{
        "variable": col,
        "non_missing": int(full_panel[col].notnull().sum()),
        "coverage_ratio": float(full_panel[col].notnull().mean()),
    } for col in candidate_feature_cols]).sort_values("coverage_ratio", ascending=False)
    selected_feature_cols = feature_coverage.loc[feature_coverage["coverage_ratio"] >= PARAMS["coverage_threshold"], "variable"].tolist()
    dropped_low_coverage = feature_coverage.loc[feature_coverage["coverage_ratio"] < PARAMS["coverage_threshold"], "variable"].tolist()
    feature_coverage.to_csv(OUTPUT_DIR / "driver_feature_coverage.csv", index=False, encoding="utf-8-sig")
    full_panel[["date"] + target_cols_all + selected_feature_cols].to_csv(OUTPUT_DIR / "driver_raw_panel_selected_features.csv", index=False, encoding="utf-8-sig")

    ltc_sensitivity = compute_ltc_window_sensitivity()

    external_features, engineered_external_cols = build_external_features(full_panel, selected_feature_cols, PARAMS["feature_lags"], PARAMS["feature_rolling_windows"])
    model_base = full_panel[["date"] + target_cols_all].merge(external_features, on="date", how="left")
    model_base = model_base.replace([np.inf, -np.inf], np.nan)
    model_base = model_base.dropna(subset=MAIN_TARGET_COLS + engineered_external_cols).reset_index(drop=True)
    feature_group_map = {col: infer_feature_group(col) for col in engineered_external_cols}

    pd.DataFrame({"variable": selected_feature_cols, "group": [infer_feature_group(col) for col in selected_feature_cols]}).to_csv(OUTPUT_DIR / "driver_selected_feature_dictionary.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame({"feature": engineered_external_cols, "group": [feature_group_map[col] for col in engineered_external_cols]}).to_csv(OUTPUT_DIR / "driver_engineered_feature_dictionary.csv", index=False, encoding="utf-8-sig")
    model_base.to_csv(OUTPUT_DIR / "driver_feature_dataset.csv", index=False, encoding="utf-8-sig")

    target_variants = model_base[["date"] + MAIN_TARGET_COLS + [LTC_TARGET_COL]].copy()
    for target_col in MAIN_TARGET_COLS:
        target_variants["{}_delta20".format(target_col)] = target_variants[target_col] - target_variants[target_col].shift(PARAMS["delta_window"])
    target_variants.to_csv(OUTPUT_DIR / "target_variants_dataset.csv", index=False, encoding="utf-8-sig")

    regression_rows, residual_rows, regression_importance_rows = [], [], []
    for target_col in MAIN_TARGET_COLS:
        print("Regression targets:", target_col)
        model_base["{}_level_y".format(target_col)] = model_base[target_col]
        model_base["{}_delta20_y".format(target_col)] = model_base[target_col] - model_base[target_col].shift(PARAMS["delta_window"])
        rows, imp = run_level_or_delta_regression(model_base, target_col, "level", "{}_level_y".format(target_col), engineered_external_cols, feature_group_map)
        regression_rows.extend(rows)
        regression_importance_rows.extend(imp)
        rows, imp = run_level_or_delta_regression(model_base, target_col, "delta20", "{}_delta20_y".format(target_col), engineered_external_cols, feature_group_map)
        regression_rows.extend(rows)
        regression_importance_rows.extend(imp)
        rows, residual_detail, imp = run_ar_residual_regression(model_base, target_col, engineered_external_cols, feature_group_map)
        regression_rows.extend(rows)
        residual_rows.extend(residual_detail)
        regression_importance_rows.extend(imp)

    regression_detail, regression_summary = summarize_regression(regression_rows)
    regression_detail.to_csv(OUTPUT_DIR / "regression_model_performance_by_target_type.csv", index=False, encoding="utf-8-sig")
    regression_summary.to_csv(OUTPUT_DIR / "regression_model_performance_summary.csv", index=False, encoding="utf-8-sig")

    ar_base = regression_summary.loc[regression_summary["model"].eq("AR_Ridge"), ["target", "target_type", "r2_mean", "rmse_mean", "mae_mean"]].rename(columns={"r2_mean": "ar_r2_mean", "rmse_mean": "ar_rmse_mean", "mae_mean": "ar_mae_mean"})
    regression_incremental = regression_summary.merge(ar_base, on=["target", "target_type"], how="left")
    regression_incremental["incremental_r2_vs_ar"] = regression_incremental["r2_mean"] - regression_incremental["ar_r2_mean"]
    regression_incremental["rmse_reduction_vs_ar"] = regression_incremental["ar_rmse_mean"] - regression_incremental["rmse_mean"]
    regression_incremental["mae_reduction_vs_ar"] = regression_incremental["ar_mae_mean"] - regression_incremental["mae_mean"]
    regression_incremental.to_csv(OUTPUT_DIR / "regression_incremental_vs_ar.csv", index=False, encoding="utf-8-sig")

    residual_driver_performance = pd.DataFrame(residual_rows)
    residual_driver_performance.to_csv(OUTPUT_DIR / "residual_driver_performance.csv", index=False, encoding="utf-8-sig")

    driver_importance_regression_long = pd.concat(regression_importance_rows, ignore_index=True)
    driver_importance_regression_long.insert(0, "analysis_type", "regression")
    driver_importance_regression_long.to_csv(OUTPUT_DIR / "driver_importance_regression_long.csv", index=False, encoding="utf-8-sig")

    classification_rows, threshold_rows, classification_importance_rows = [], [], []
    for target_col in MAIN_TARGET_COLS:
        print("Classification targets:", target_col)
        rows, thresholds, imp = run_high_state_classification(model_base, target_col, engineered_external_cols, feature_group_map)
        classification_rows.extend(rows)
        threshold_rows.extend(thresholds)
        classification_importance_rows.extend(imp)

    classification_detail = pd.DataFrame(classification_rows)
    classification_detail.to_csv(OUTPUT_DIR / "classification_model_performance_by_fold.csv", index=False, encoding="utf-8-sig")
    classification_summary = classification_detail.groupby(["target", "high_state", "model_family", "model", "feature_set"])[["auc", "average_precision", "balanced_accuracy", "f1", "accuracy", "positive_rate_test", "positive_rate_pred"]].agg(["mean", "std"])
    classification_summary.columns = ["{}_{}".format(a, b) for a, b in classification_summary.columns]
    classification_summary = classification_summary.reset_index()
    classification_summary.to_csv(OUTPUT_DIR / "classification_model_performance.csv", index=False, encoding="utf-8-sig")
    pd.DataFrame(threshold_rows).to_csv(OUTPUT_DIR / "classification_thresholds_by_fold.csv", index=False, encoding="utf-8-sig")

    driver_importance_classification_long = pd.concat(classification_importance_rows, ignore_index=True)
    driver_importance_classification_long.insert(0, "analysis_type", "classification")
    driver_importance_classification_long.to_csv(OUTPUT_DIR / "driver_importance_classification_long.csv", index=False, encoding="utf-8-sig")

    driver_group_importance_summary = pd.concat([driver_importance_regression_long, driver_importance_classification_long], ignore_index=True, sort=False)
    driver_group_importance_summary["positive_sum"] = driver_group_importance_summary.groupby(["analysis_type", "target", "mode"])["importance_positive"].transform("sum")
    driver_group_importance_summary["normalized_importance"] = np.where(driver_group_importance_summary["positive_sum"] > 0, driver_group_importance_summary["importance_positive"] / driver_group_importance_summary["positive_sum"], 0.0)
    driver_group_importance_summary.to_csv(OUTPUT_DIR / "driver_group_importance_summary.csv", index=False, encoding="utf-8-sig")

    # Backward-compatible files for earlier analysis references.
    level_summary = regression_summary[regression_summary["target_type"] == "level"].copy()
    level_summary.drop(columns=["target_type"]).to_csv(OUTPUT_DIR / "model_performance_summary.csv", index=False, encoding="utf-8-sig")
    regression_incremental[regression_incremental["target_type"] == "level"].drop(columns=["target_type"]).to_csv(OUTPUT_DIR / "incremental_r2_by_target.csv", index=False, encoding="utf-8-sig")
    driver_importance_regression_long.to_csv(OUTPUT_DIR / "permutation_importance_long.csv", index=False, encoding="utf-8-sig")
    driver_group_importance_summary.to_csv(OUTPUT_DIR / "group_importance_summary.csv", index=False, encoding="utf-8-sig")

    plot_analysis_outputs(regression_summary, regression_incremental, residual_driver_performance, classification_summary, driver_group_importance_summary)

    manifest = {
        "method_note": "External driver analysis for R2G dynamic TCI. Main driver targets are level, delta20, fold-wise AR residuals, and high-connectedness states. Same-scale LTC under the 252-day rolling window is diagnosed separately and excluded from main driver targets.",
        "main_targets": MAIN_TARGET_COLS,
        "excluded_from_main_driver_model": [LTC_TARGET_COL],
        "target_types": ["level", "delta20", "ar_residual", "high_q75", "high_q90"],
        "selected_raw_feature_count": int(len(selected_feature_cols)),
        "engineered_external_feature_count": int(len(engineered_external_cols)),
        "date_range_model_dataset": [str(model_base["date"].min().date()), str(model_base["date"].max().date())],
        "dropped_low_coverage_features": dropped_low_coverage,
        "cftc_precious_rows": int(cftc_precious_rows),
        "outputs": {
            "target_variants_dataset": str((OUTPUT_DIR / "target_variants_dataset.csv").relative_to(PROJECT_ROOT)),
            "regression_model_performance_by_target_type": str((OUTPUT_DIR / "regression_model_performance_by_target_type.csv").relative_to(PROJECT_ROOT)),
            "regression_incremental_vs_ar": str((OUTPUT_DIR / "regression_incremental_vs_ar.csv").relative_to(PROJECT_ROOT)),
            "residual_driver_performance": str((OUTPUT_DIR / "residual_driver_performance.csv").relative_to(PROJECT_ROOT)),
            "classification_model_performance": str((OUTPUT_DIR / "classification_model_performance.csv").relative_to(PROJECT_ROOT)),
            "classification_thresholds_by_fold": str((OUTPUT_DIR / "classification_thresholds_by_fold.csv").relative_to(PROJECT_ROOT)),
            "driver_importance_regression_long": str((OUTPUT_DIR / "driver_importance_regression_long.csv").relative_to(PROJECT_ROOT)),
            "driver_importance_classification_long": str((OUTPUT_DIR / "driver_importance_classification_long.csv").relative_to(PROJECT_ROOT)),
            "driver_group_importance_summary": str((OUTPUT_DIR / "driver_group_importance_summary.csv").relative_to(PROJECT_ROOT)),
            "ltc_window_sensitivity": str((OUTPUT_DIR / "ltc_window_sensitivity.csv").relative_to(PROJECT_ROOT)),
            "regression_incremental_r2_heatmap": str((FIG_DIR / "regression_incremental_r2_heatmap.png").relative_to(PROJECT_ROOT)),
            "delta20_driver_group_heatmap": str((FIG_DIR / "delta20_driver_group_heatmap.png").relative_to(PROJECT_ROOT)),
            "ar_residual_driver_group_heatmap": str((FIG_DIR / "ar_residual_driver_group_heatmap.png").relative_to(PROJECT_ROOT)),
            "high_state_auc_heatmap": str((FIG_DIR / "high_state_auc_heatmap.png").relative_to(PROJECT_ROOT)),
            "high_state_driver_group_heatmap": str((FIG_DIR / "high_state_driver_group_heatmap.png").relative_to(PROJECT_ROOT)),
            "ltc_window_sensitivity_plot": str((FIG_DIR / "ltc_window_sensitivity.png").relative_to(PROJECT_ROOT)),
        },
    }
    with open(OUTPUT_DIR / "driver_factor_analysis_manifest.json", "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
    return manifest


## 9. 执行分析


In [9]:
manifest = run_analysis()


Regression targets: tci_same_scale_STC
Regression targets: tci_same_scale_MTC
Regression targets: tci_within_gold
Regression targets: tci_within_silver
Regression targets: tci_within_platinum
Regression targets: tci_within_palladium
Classification targets: tci_same_scale_STC


C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\m

Classification targets: tci_same_scale_MTC


C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarnin

Classification targets: tci_within_gold


...

Classification targets: tci_within_platinum


C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no true samples.
  'recall', 'true', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1137: UndefinedMetricWarnin

Classification targets: tci_within_palladium


C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\metrics\classification.py:1135: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 due to no predicted samples.
  'precision', 'predicted', average, warn_for)
C:\Users\Administrator\Anaconda3\lib\site-packages\sklearn\m

{
  "method_note": "External driver analysis for R2G dynamic TCI. Main driver targets are level, delta20, fold-wise AR residuals, and high-connectedness states. Same-scale LTC under the 252-day rolling window is diagnosed separately and excluded from main driver targets.",
  "main_targets": [
    "tci_same_scale_STC",
    "tci_same_scale_MTC",
    "tci_within_gold",
    "tci_within_silver",
    "tci_within_platinum",
    "tci_within_palladium"
  ],
  "excluded_from_main_driver_model": [
    "tci_same_scale_LTC"
  ],
  "target_types": [
    "level",
    "delta20",
    "ar_residual",
    "high_q75",
    "high_q90"
  ],
  "selected_raw_feature_count": 59,
  "engineered_external_feature_count": 295,
  "date_range_model_dataset": [
    "2013-03-04",
    "2026-03-31"
  ],
  "dropped_low_coverage_features": [
    "credit_spread_hy",
    "credit_spread_ig"
  ],
  "cftc_precious_rows": 4152,
  "outputs": {
    "target_variants_dataset": "data\\processed\\08_tci_external_driver_factor_analysis\\

## 10. 输出文件

主要结果保存在 `data/processed/08_tci_external_driver_factor_analysis`。新增输出包括：

- `target_variants_dataset.csv`
- `regression_model_performance_by_target_type.csv`
- `regression_incremental_vs_ar.csv`
- `residual_driver_performance.csv`
- `classification_model_performance.csv`
- `classification_thresholds_by_fold.csv`
- `driver_importance_regression_long.csv`
- `driver_importance_classification_long.csv`
- `driver_group_importance_summary.csv`
- `visualizations/regression_incremental_r2_heatmap.png`
- `visualizations/delta20_driver_group_heatmap.png`
- `visualizations/ar_residual_driver_group_heatmap.png`
- `visualizations/high_state_auc_heatmap.png`
- `visualizations/high_state_driver_group_heatmap.png`
